# Open Library｜作家・作品・版の記録を調べる
**技術講習会「文学研究のためのデジタル資料活用」｜2026年10月16日**

Open Libraryでは、作家の登録情報を**Author**、作品をまとめる登録情報を**Work**、個々の版・刊行物の登録情報を**Edition**と呼びます。

このNotebookでは、**WorkごとにEditionが何件登録されているか**を比較します。Editionの登録件数は、発行部数・販売部数・読者数ではありません。

**上から順に、必要なセルの左側の ▶ を押して実行します。** 「すべてのセルを実行」は必要ありません。

- **前半：Conradの8作品を表示** — 保存済みの数値だけを使います。講習会用サーバーにはアクセスしません。
- **後半：作家や作品を自由検索** — Open Libraryの保存済みデータを検索する講習会用サーバーにアクセスします。

## 1. Joseph Conradの8作品を比較する

まず、あらかじめ保存してある8作品について、WorkごとのEdition登録件数を表示します。

ここにあるのは**Open Libraryの公開ページで確認して記録した参考値（2026年10月14日）**です。後半の自由検索で使う**2026年2月28日版のデータ**とは件数が異なる場合があります。

次のセルを実行しても、講習会用APIへの通信は発生しません。

In [ ]:
# @title Conradの8作品を表で表示する（サーバーへの通信なし） {display-mode: "form"}
import pandas as pd
from IPython.display import display

# Open Library公開ページで確認した参考値（2026年10月14日）。
# このセルはインターネットや講習会用APIにアクセスしません。
CONRAD_EIGHT = [
    {"work_id": "OL38630W", "title": "Almayer's Folly", "editions": 97, "year": 1895},
    {"work_id": "OL38815W", "title": 'The Nigger of the "Narcissus"', "editions": 93, "year": 1897},
    {"work_id": "OL38663W", "title": "Heart of Darkness", "editions": 553, "year": 1899},
    {"work_id": "OL38684W", "title": "Lord Jim", "editions": 345, "year": 1900},
    {"work_id": "OL38700W", "title": "Nostromo", "editions": 255, "year": 1904},
    {"work_id": "OL39108W", "title": "The Secret Agent", "editions": 335, "year": 1907},
    {"work_id": "OL38763W", "title": "Victory", "editions": 146, "year": 1915},
    {"work_id": "OL38827W", "title": "The Shadow-Line", "editions": 147, "year": 1917},
]

conrad_df = pd.DataFrame(CONRAD_EIGHT).sort_values("year").reset_index(drop=True)
display(conrad_df.rename(columns={
    "year": "初出・初刊年の目安",
    "title": "作品名",
    "work_id": "Work ID",
    "editions": "Edition登録件数（参考値）",
})[["初出・初刊年の目安", "作品名", "Work ID", "Edition登録件数（参考値）"]])

In [ ]:
# @title Conradの8作品をグラフで表示する（サーバーへの通信なし） {display-mode: "form"}
import matplotlib.pyplot as plt

plot_df = conrad_df.iloc[::-1]
fig, ax = plt.subplots(figsize=(11, 6))
labels = [f'{r.year}  {r.title}' for r in plot_df.itertuples()]
bars = ax.barh(labels, plot_df["editions"])
ax.bar_label(bars, padding=3)
ax.set_xlim(0, max(plot_df["editions"]) * 1.15)
ax.set_xlabel("Edition records in Open Library (saved reference values)")
ax.set_title("Joseph Conrad — 8 Work records")
fig.tight_layout()
plt.show()
print("データ：Open Library公開ページの保存済み参考値（2026年10月14日）")

## 2. 作家と作品を検索する

ここからは、講習会用サーバーに接続してOpen Libraryの登録情報を検索します。**このセルを実行したときに、初めて検索APIへの通信が発生します。**

作家名と作品名を入力して ▶ を押すと、それぞれの候補が表示されます。例えば、最初は *Joseph Conrad* と *Heart of Darkness* を設定しています。

- **Author ID** は、作家の登録番号です。
- **Work ID** は、作品の登録番号です。

作品名の検索は**著者を指定しない検索**です。同名の別作品や、一つの文学作品に対応する複数のWorkレコードが出る場合があるため、Open Libraryへのリンクも確認してください。

In [ ]:
# @title 作家名と作品名を入力して検索する（APIに接続） {display-mode: "form"}
AUTHOR_NAME = "Joseph Conrad" # @param {type:"string"}
WORK_TITLE = "Heart of Darkness" # @param {type:"string"}

# 以下は変更不要です
API_URL = "https://bunsekisan.jp/ol-workshop-api"

import time
import unicodedata
import requests
import pandas as pd
from IPython.display import display

def norm(value):
    return unicodedata.normalize("NFKC", str(value or "")).casefold().strip()

def api_get(path, params):
    # 混雑時の再試行は1回だけ
    for attempt in range(2):
        try:
            response = requests.get(API_URL + path, params=params, timeout=(5, 18))
            if response.status_code in (429, 503) and attempt == 0:
                time.sleep(3)
                continue
            response.raise_for_status()
            return response.json()
        except requests.RequestException:
            if attempt == 1:
                raise
            time.sleep(3)
    raise RuntimeError("検索サーバーに接続できません")

try:
    a = api_get("/v1/authors", {"name": AUTHOR_NAME, "limit": 12})
    w = api_get("/v1/works", {"title": WORK_TITLE, "limit": 30})
    authors = a.get("candidates", [])
    works = w.get("works", [])
    # 完全に同じタイトルを先に、次いでEdition登録件数の多い順に表示
    works.sort(key=lambda x: (norm(x.get("title")) != norm(WORK_TITLE),
                              -int(x.get("linked_edition_records", 0))))
    print("検索結果：Open Libraryの2026年2月28日版の保存データ")
except (requests.RequestException, RuntimeError, ValueError) as exc:
    authors, works = [], []
    print("検索サーバーに接続できませんでした：", exc)
    print("少し待ってこのセルを再実行してください。前半のConrad 8作品は引き続き表示できます。")

print("\n【作家の候補】")
if authors:
    display(pd.DataFrame([{
        "Author ID": x["author_id"], "登録名": x["name"],
        "関連Work件数": x.get("linked_work_records", "—"),
        "Open Library": f'https://openlibrary.org/authors/{x["author_id"]}'
    } for x in authors]).head(12))
else:
    print("候補を表示できませんでした。")

print("\n【作品の候補】（著者による絞り込みはしていません）")
if works:
    display(pd.DataFrame([{
        "Work ID": x["work_id"], "タイトル": x["title"],
        "Edition登録件数": x.get("linked_edition_records", 0),
        "Open Library": f'https://openlibrary.org/works/{x["work_id"]}'
    } for x in works]).head(15))
else:
    print("候補を表示できませんでした。")

## 3. 作家の作品群を比較する

前の一覧で確認した**Author ID**と**Work ID**を入力して ▶ を押します。初期設定はConradと *Heart of Darkness* です。

年代は空欄のままでも構いません。開始年と終了年を指定すると、その年代に対応すると登録されたWorkに絞って比較します。ただし、Open LibraryのWorkの年は、実際の初刊年と異なる場合があります。

**年代を指定しても、Edition登録件数はその年代だけの出版数ではなく、各Workに結びつく全期間のEdition登録件数です。**

In [ ]:
# @title 比較する作家・作品・年代を指定する（APIに接続） {display-mode: "form"}
AUTHOR_ID = "OL19441A" # @param {type:"string"}
WORK_ID = "OL38663W" # @param {type:"string"}
YEAR_FROM = "" # @param {type:"string"}
YEAR_TO = "" # @param {type:"string"}
DISPLAY_LIMIT = 15 # @param {type:"integer"}

if not (1 <= int(DISPLAY_LIMIT) <= 50):
    raise ValueError("表示件数は1〜50にしてください。")
if bool(YEAR_FROM.strip()) != bool(YEAR_TO.strip()):
    raise ValueError("年代を絞る場合は開始年と終了年の両方を入力してください。")
if YEAR_FROM.strip() and (not YEAR_FROM.isdecimal() or not YEAR_TO.isdecimal()
                          or int(YEAR_FROM) > int(YEAR_TO)):
    raise ValueError("年代を数字で正しく指定してください。")

api_df = pd.DataFrame(columns=["work_id", "title", "linked_edition_records"])
try:
    params = {
        "author_ids": AUTHOR_ID.strip(),
        "target_work_id": WORK_ID.strip(),
        "display_limit": int(DISPLAY_LIMIT),
    }
    if YEAR_FROM.strip():
        params.update({"year_from": int(YEAR_FROM), "year_to": int(YEAR_TO)})
    result = api_get("/v1/collections/by-authors", params)
    s = result["statistics_full_matching_set"]
    print("【Open Library 2026年2月28日版】")
    print(f'比較対象：{s["work_records"]:,} Workレコード')
    print(f'Edition登録件数の中央値：{s["median_linked_edition_records"]}件')
    target = s.get("target")
    if target:
        print(f'指定したWork：{target["title"]}（{target["work_id"]}）')
        print(f'Edition登録件数：{target["linked_edition_records"]:,}件')
        print(f'順位：{target["rank_competition"]:,}位／{target["rank_denominator_work_records"]:,}件（同数同順位）')
    elif WORK_ID.strip():
        print("指定したWork IDは、この作家・年代の集合に含まれていません。")
    api_df = pd.DataFrame(result.get("works_displayed", []))
    print(f'\n【表示するWork：{len(api_df)}件】')
    if not api_df.empty:
        display(api_df[["work_id", "title", "linked_edition_records"]].rename(columns={
            "work_id": "Work ID", "title": "作品名", "linked_edition_records": "Edition登録件数"
        }).reset_index(drop=True))
    else:
        print("表示対象はありません。")
except (requests.RequestException, RuntimeError, KeyError, ValueError) as exc:
    print("比較データを取得できませんでした：", exc)
    print("少し待ってこのセルを再実行してください。前半のConradのグラフはそのまま使えます。")

In [ ]:
# @title 自由検索したWorkをグラフで表示する（このセルは通信なし） {display-mode: "form"}
import matplotlib.pyplot as plt

if api_df.empty:
    print("比較結果がありません。直前のセルを実行してください。")
else:
    plot_api = api_df.iloc[::-1]
    labels = [f'{row.title} [{row.work_id}]' for row in plot_api.itertuples()]
    fig, ax = plt.subplots(figsize=(11, max(4, len(plot_api) * 0.48 + 1)))
    bars = ax.barh(labels, plot_api["linked_edition_records"])
    ax.bar_label(bars, padding=3)
    ax.set_xlim(0, max(plot_api["linked_edition_records"].max() * 1.15, 1))
    ax.set_xlabel("Edition records in Open Library (2026-02-28 snapshot)")
    ax.set_title("Edition records by Work (selected author)")
    fig.tight_layout()
    plt.show()

---
**数値について**　Edition登録件数は、出版部数・読者数・文学的評価を表すものではありません。また、同じ文学作品が複数のWorkレコードに分かれている場合があります。後半の順位・中央値は、Open Libraryに登録された**Workレコード単位**の数値です。

**使用したデータ**　前半：Open Library公開ページで確認したConrad 8作品の保存済み参考値（2026年10月14日）。後半：Open LibraryのWorks・Authors・Editionsの**2026年2月28日版**を使った講習会用検索データベース（未監査の登録情報を含みます）。取得時点が違うため件数が一致しない場合があります。

**原題表記**　*The Nigger of the “Narcissus”* は1897年の原題を使用していますが、現在では差別的とされる語を含みます。

[Open Library：Joseph Conrad](https://openlibrary.org/authors/OL19441A) · [*Heart of Darkness*](https://openlibrary.org/works/OL38663W)